# 🛠️ Servidor MCP (FastMCP) + Cliente OpenAI — Monedas y Clima

**Reto proyecto — Desarrollo de Soluciones IA**

Servidor **MCP con FastMCP** que expone **5 herramientas** sobre 2 APIs públicas (**ExchangeRate-API** para divisas y **Open-Meteo** para geocodificación + clima), y un **cliente OpenAI** (tu recurso de Azure, `gpt-5`) que descubre esas herramientas y las usa con *function calling*, encadenando llamadas (geocodificación → clima). Incluye una **CLI** en lenguaje natural.

### Mapeo de la estructura pedida → celdas

```
├── server/api_clients.py       → Celda «Clientes de APIs»
│   ├── currency_tools.py       → Celda «Herramientas de monedas»
│   ├── geocoding_tools.py      → Celda «Geocodificación»
│   ├── weather_tools.py        → Celda «Herramientas de clima»
│   └── mcp_server.py           → Celda «Servidor MCP»
├── client/openai_client.py     → Celda «Cliente OpenAI con MCP»
│   └── cli_interface.py        → Celda «Interfaz CLI»
├── config/settings.py          → Celda «Configuración»
├── main_server.py              → Celda «Arrancar servidor en localhost»
├── main_client.py              → Celda «Opción B (run_cli)»
├── requirements.txt            → Celda de dependencias
├── .env                        → Credenciales (privadas)
└── README.md                   → Este encabezado
```

### Las 5 herramientas

1. `convert_currency(amount, from_currency, to_currency)` — conversión entre divisas (ExchangeRate).
2. `get_exchange_rates(base_currency)` — tasas de una moneda base frente a muchas (ExchangeRate).
3. `geocode_city(city)` — ciudad → lat/lon, país, zona horaria (Open-Meteo).
4. `get_current_weather(latitude, longitude)` — clima actual (Open-Meteo).
5. `get_weather_forecast(latitude, longitude, days)` — pronóstico de varios días (Open-Meteo).

### Configuración (`.env`)

```dotenv
# OpenAI vía tu recurso de Azure (cliente)
AZURE_OPENAI_ENDPOINT=https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/
AZURE_OPENAI_API_KEY=tu_clave_de_azure
OPENAI_MODEL=gpt-5

# ExchangeRate-API (clave GRATUITA: https://www.exchangerate-api.com/)
API_KEY_EXCHANGE=tu_api_key_de_exchangerate

# Open-Meteo: NO necesita clave
MCP_HOST=127.0.0.1
MCP_PORT=8000
```

> **Open-Meteo es gratis y sin clave** (geocodificación + clima). **ExchangeRate sí necesita** una API key gratuita (regístrate en exchangerate-api.com); sin ella, las 2 herramientas de divisas devolverán un error controlado, pero las 3 de clima funcionan igual.

### Cómo ejecutar
Ejecuta las celdas de arriba abajo: instala dependencias, carga config, define herramientas, **arranca el servidor en localhost** (sección 8), crea el cliente y usa la **Opción A** (consultas directas) o la **Opción B** (CLI interactiva).


## 1. Dependencias (`requirements.txt`)

```text
fastmcp
openai
requests
python-dotenv
uvicorn
```


In [40]:
# Instalación de dependencias (ejecutar una vez)
import sys, subprocess

paquetes = ["fastmcp", "openai", "requests", "python-dotenv", "uvicorn"]
# En Windows, el paquete `mcp` (que usa FastMCP) requiere pywin32.
if sys.platform == "win32":
    paquetes.append("pywin32>=311")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *paquetes], check=True)
print("✅ Dependencias instaladas.")


✅ Dependencias instaladas.


## 2. Configuración (`config/settings.py`)

Carga las variables del `.env`: credenciales de Azure para el cliente OpenAI, la clave de
ExchangeRate-API, las URLs de las APIs y el host/puerto del servidor MCP.
IPython >= 7 (Jupyter moderno) soporta `await` en el nivel superior de las celdas de forma
nativa, por lo que **no es necesario** `nest_asyncio`.


In [41]:
import os
from dotenv import load_dotenv

load_dotenv()

# --- OpenAI vía tu recurso de Azure (para el cliente) ---
_raw = os.getenv("AZURE_OPENAI_ENDPOINT",
                 "https://marcvancutseme7172-2656-resource.services.ai.azure.com/")
_base = _raw.rstrip("/")
for _suf in ("/openai/v1", "/openai"):
    if _base.endswith(_suf):
        _base = _base[: -len(_suf)]
OPENAI_BASE_URL = _base + "/openai/v1/"
OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5")

# --- ExchangeRate-API (clave gratuita) ---
API_KEY_EXCHANGE = os.getenv("API_KEY_EXCHANGE", "")
BASE_URL_EXCHANGE = os.getenv("BASE_URL_EXCHANGE", "https://v6.exchangerate-api.com/v6")

# --- Open-Meteo (sin clave) ---
BASE_URL_WEATHER = os.getenv("BASE_URL_WEATHER", "https://api.open-meteo.com/v1")
BASE_URL_GEOCODING = os.getenv("BASE_URL_GEOCODING",
                               "https://geocoding-api.open-meteo.com/v1/search")

# --- Servidor MCP (localhost) ---
MCP_HOST = os.getenv("MCP_HOST", "127.0.0.1")
MCP_PORT = int(os.getenv("MCP_PORT", "8000"))
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"

# --- Límites del cliente OpenAI (configurables por entorno) ---
MAX_PASOS_HERRAMIENTAS = int(os.getenv("MCP_MAX_PASOS", "8"))     # nº máx. de pasos de tool-calling
MAX_REINTENTOS_OPENAI = int(os.getenv("MCP_MAX_REINTENTOS", "2")) # reintentos ante fallos de red

print("✅ Configuración cargada.")
print(f"   OpenAI:  {OPENAI_BASE_URL}  | modelo: {OPENAI_MODEL}  | clave: {'✅' if OPENAI_API_KEY else '❌'}")
print(f"   Divisas: ExchangeRate key {'✅' if API_KEY_EXCHANGE else '❌ (consíguela en exchangerate-api.com)'}")
print(f"   MCP:     {MCP_URL}")


✅ Configuración cargada.
   OpenAI:  https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/  | modelo: gpt-5  | clave: ✅
   Divisas: ExchangeRate key ✅
   MCP:     http://127.0.0.1:8000/mcp


## 3. Clientes de APIs externas (`server/api_clients.py`)

Helper común para llamadas HTTP con **manejo de errores** (timeouts, códigos de estado, JSON inválido) y **logging**. Lo usan todas las herramientas.


In [42]:
import logging
import requests

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("api_clients")


class APIError(Exception):
    """Error controlado al llamar a una API externa."""


def http_get_json(url: str, params: dict | None = None, timeout: int = 10) -> dict:
    """Hace un GET y devuelve el JSON, o lanza APIError con un mensaje claro."""
    url_log = url
    try:  # no registrar la API key de ExchangeRate si aparece en la URL
        if API_KEY_EXCHANGE and API_KEY_EXCHANGE in url_log:
            url_log = url_log.replace(API_KEY_EXCHANGE, "***")
    except NameError:
        pass
    logger.info("GET %s params=%s", url_log, params or {})
    try:
        resp = requests.get(url, params=params, timeout=timeout)
    except requests.exceptions.Timeout:
        raise APIError(f"Tiempo de espera agotado al llamar a {url}")
    except requests.exceptions.RequestException as e:
        raise APIError(f"Error de red al llamar a la API: {e}")

    if resp.status_code != 200:
        detalle = " ".join(resp.text.split())[:200]   # una línea, truncado para logs limpios
        logger.warning("Respuesta %s de %s: %s", resp.status_code, url, detalle)
        raise APIError(f"La API respondió con código {resp.status_code}: {detalle}")
    try:
        return resp.json()
    except ValueError:
        raise APIError("La respuesta de la API no es un JSON válido.")


## 4. Herramientas de monedas (`server/currency_tools.py`)

Dos herramientas sobre **ExchangeRate-API** con parámetros tipados, validación de códigos de moneda (3 letras ISO), docstrings en español y manejo de errores.


In [43]:
from typing import Any
import re

# Lista informativa de códigos de moneda habituales (para el comando /monedas)
MONEDAS_COMUNES = ["USD", "EUR", "GBP", "JPY", "CHF", "CAD", "AUD", "CNY",
                   "MXN", "BRL", "ARS", "INR", "RUB", "SEK", "NOK", "PLN"]


def _validar_moneda(codigo: str) -> str:
    codigo = (codigo or "").strip().upper()
    if not re.fullmatch(r"[A-Z]{3}", codigo):
        raise ValueError(f"Código de moneda no válido: '{codigo}'. Usa 3 letras ISO (p. ej. USD, EUR).")
    return codigo


def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict[str, Any]:
    """Convierte una cantidad de una moneda a otra usando las tasas actuales (ExchangeRate-API).

    Args:
        amount: cantidad a convertir (número positivo).
        from_currency: código ISO 4217 de la moneda de origen (p. ej. 'USD').
        to_currency: código ISO 4217 de la moneda de destino (p. ej. 'EUR').

    Returns:
        dict con las claves:
          - amount (float): cantidad original.
          - from_currency (str): código de la moneda de origen (normalizado a mayúsculas).
          - to_currency (str): código de la moneda de destino (normalizado a mayúsculas).
          - rate (float): tasa de cambio aplicada.
          - converted_amount (float): resultado de la conversión.
    """
    try:
        cantidad = float(amount)
    except (TypeError, ValueError):
        raise ValueError("La cantidad debe ser un número.")
    if cantidad <= 0:
        raise ValueError("La cantidad debe ser un número positivo.")
    origen = _validar_moneda(from_currency)
    destino = _validar_moneda(to_currency)

    if not API_KEY_EXCHANGE:
        raise APIError("Falta la clave API_KEY_EXCHANGE. Consíguela gratis en exchangerate-api.com.")

    url = f"{BASE_URL_EXCHANGE}/{API_KEY_EXCHANGE}/pair/{origen}/{destino}/{cantidad}"
    logger.info("convert_currency: %s %s -> %s", cantidad, origen, destino)
    data = http_get_json(url)
    if data.get("result") != "success":
        raise APIError(f"ExchangeRate-API devolvió un error: {data.get('error-type', 'desconocido')}")
    return {
        "amount": cantidad,
        "from_currency": origen,
        "to_currency": destino,
        "rate": data.get("conversion_rate"),
        "converted_amount": data.get("conversion_result"),
    }


def get_exchange_rates(base_currency: str) -> dict[str, Any]:
    """Devuelve las tasas de cambio de una moneda base frente a múltiples monedas (ExchangeRate-API).

    Args:
        base_currency: código ISO 4217 de la moneda base (p. ej. 'EUR').

    Returns:
        dict con las claves:
          - base_currency (str): moneda base (normalizada a mayúsculas).
          - rates (dict[str, float]): diccionario {codigo_moneda: tasa} para cada moneda destino.
    """
    base = _validar_moneda(base_currency)
    if not API_KEY_EXCHANGE:
        raise APIError("Falta la clave API_KEY_EXCHANGE. Consíguela gratis en exchangerate-api.com.")
    url = f"{BASE_URL_EXCHANGE}/{API_KEY_EXCHANGE}/latest/{base}"
    logger.info("get_exchange_rates: base=%s", base)
    data = http_get_json(url)
    if data.get("result") != "success":
        raise APIError(f"ExchangeRate-API devolvió un error: {data.get('error-type', 'desconocido')}")
    return {"base_currency": base, "rates": data.get("conversion_rates", {})}


def salud_exchange(timeout: int = 5) -> tuple[bool, str]:
    """Comprueba si ExchangeRate-API está operativa (llamada de prueba ligera).

    Returns:
        (operativa, detalle): booleano y un mensaje explicativo.
    """
    if not API_KEY_EXCHANGE:
        return False, "falta API_KEY_EXCHANGE"
    try:
        data = http_get_json(f"{BASE_URL_EXCHANGE}/{API_KEY_EXCHANGE}/latest/USD", timeout=timeout)
        if data.get("result") == "success":
            return True, "operativa"
        return False, data.get("error-type", "error desconocido")
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


## 5. Geocodificación (`server/geocoding_tools.py`)

Convierte un nombre de ciudad en coordenadas usando la API de geocodificación de **Open-Meteo** (sin clave). Es el primer paso del flujo *ciudad → clima*.


In [44]:
from typing import Any


def geocode_city(city: str) -> dict[str, Any]:
    """Geocodifica un nombre de ciudad a coordenadas, país y zona horaria (Open-Meteo).

    Args:
        city: nombre de la ciudad (p. ej. 'Madrid').

    Returns:
        dict con las claves:
          - city (str): nombre de la ciudad encontrada.
          - country (str): país al que pertenece.
          - latitude (float): latitud en grados decimales.
          - longitude (float): longitud en grados decimales.
          - timezone (str): zona horaria IANA (p. ej. 'Europe/Madrid').
    """
    nombre = (city or "").strip()
    if len(nombre) < 2:
        raise ValueError("El nombre de la ciudad es demasiado corto.")
    logger.info("geocode_city: %s", nombre)
    data = http_get_json(BASE_URL_GEOCODING,
                         params={"name": nombre, "count": 1, "language": "es", "format": "json"})
    resultados = data.get("results") or []
    if not resultados:
        raise APIError(f"No se encontró la ciudad '{nombre}'. Revisa el nombre.")
    r = resultados[0]
    return {
        "city": r.get("name"),
        "country": r.get("country"),
        "latitude": r.get("latitude"),
        "longitude": r.get("longitude"),
        "timezone": r.get("timezone"),
    }


## 6. Herramientas de clima (`server/weather_tools.py`)

Clima actual y pronóstico de varios días a partir de coordenadas, usando **Open-Meteo** (sin clave). Los códigos meteorológicos se traducen a descripciones en español.


In [45]:
from typing import Any

WEATHER_CODES = {
    0: "Despejado", 1: "Mayormente despejado", 2: "Parcialmente nublado", 3: "Nublado",
    45: "Niebla", 48: "Niebla con escarcha",
    51: "Llovizna ligera", 53: "Llovizna moderada", 55: "Llovizna densa",
    61: "Lluvia ligera", 63: "Lluvia moderada", 65: "Lluvia fuerte",
    71: "Nieve ligera", 73: "Nieve moderada", 75: "Nieve fuerte",
    80: "Chubascos ligeros", 81: "Chubascos moderados", 82: "Chubascos violentos",
    95: "Tormenta", 96: "Tormenta con granizo", 99: "Tormenta fuerte con granizo",
}

DIAS_PRONOSTICO_MIN = 1
DIAS_PRONOSTICO_MAX = 16
DIAS_PRONOSTICO_DEFECTO = 3


def _descripcion(code) -> str:
    return WEATHER_CODES.get(code, f"Código meteorológico {code}")


def _validar_coords(latitude, longitude) -> tuple[float, float]:
    try:
        lat, lon = float(latitude), float(longitude)
    except (TypeError, ValueError):
        raise ValueError("Las coordenadas deben ser números.")
    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        raise ValueError("Coordenadas fuera de rango (lat: -90..90, lon: -180..180).")
    # Limitamos la precisión a 4 decimales (~11 m); más precisión no aporta al clima
    return round(lat, 4), round(lon, 4)


def _normalizar_dias(days) -> int:
    """Normaliza el número de días al rango válido en lugar de lanzar error."""
    try:
        dias = int(days)
    except (TypeError, ValueError):
        return DIAS_PRONOSTICO_DEFECTO
    # Ajustamos al rango permitido (clamp) en vez de fallar
    return max(DIAS_PRONOSTICO_MIN, min(dias, DIAS_PRONOSTICO_MAX))


def get_current_weather(latitude: float, longitude: float) -> dict[str, Any]:
    """Devuelve el clima actual para unas coordenadas (Open-Meteo).

    Args:
        latitude: latitud en grados decimales (-90..90).
        longitude: longitud en grados decimales (-180..180).

    Returns:
        dict con las claves:
          - temperature_c (float): temperatura actual en °C.
          - humidity_pct (int): humedad relativa en %.
          - wind_kmh (float): velocidad del viento en km/h.
          - weather_code (int): código meteorológico WMO.
          - description (str): descripción en español del código meteorológico.
    """
    lat, lon = _validar_coords(latitude, longitude)
    logger.info("get_current_weather: %.4f, %.4f", lat, lon)
    data = http_get_json(f"{BASE_URL_WEATHER}/forecast", params={
        "latitude": lat, "longitude": lon,
        "current": "temperature_2m,relative_humidity_2m,weather_code,wind_speed_10m",
        "timezone": "auto",
    })
    c = data.get("current", {})
    return {
        "temperature_c": c.get("temperature_2m"),
        "humidity_pct": c.get("relative_humidity_2m"),
        "wind_kmh": c.get("wind_speed_10m"),
        "weather_code": c.get("weather_code"),
        "description": _descripcion(c.get("weather_code")),
    }


def get_weather_forecast(latitude: float, longitude: float, days: int = 3) -> dict[str, Any]:
    """Devuelve el pronóstico de varios días para unas coordenadas (Open-Meteo).

    Args:
        latitude: latitud en grados decimales (-90..90).
        longitude: longitud en grados decimales (-180..180).
        days: número de días a pronosticar. Se ajusta automáticamente al rango 1-16
            (valor por defecto 3 si se pasa algo no numérico).

    Returns:
        dict con la clave:
          - days (list[dict]): lista de días, cada uno con:
              - date (str): fecha en formato ISO 'AAAA-MM-DD'.
              - temp_max_c (float): temperatura máxima en °C.
              - temp_min_c (float): temperatura mínima en °C.
              - description (str): descripción en español del código meteorológico.
    """
    lat, lon = _validar_coords(latitude, longitude)
    dias = _normalizar_dias(days)
    logger.info("get_weather_forecast: %.4f, %.4f (%d días)", lat, lon, dias)
    data = http_get_json(f"{BASE_URL_WEATHER}/forecast", params={
        "latitude": lat, "longitude": lon,
        "daily": "temperature_2m_max,temperature_2m_min,weather_code",
        "forecast_days": dias, "timezone": "auto",
    })
    d = data.get("daily", {})
    fechas = d.get("time", [])
    pronostico = []
    for i, fecha in enumerate(fechas):
        pronostico.append({
            "date": fecha,
            "temp_max_c": d.get("temperature_2m_max", [None] * len(fechas))[i],
            "temp_min_c": d.get("temperature_2m_min", [None] * len(fechas))[i],
            "description": _descripcion(d.get("weather_code", [None] * len(fechas))[i]),
        })
    return {"days": pronostico}


## 7. Servidor MCP (`server/mcp_server.py`)

Creamos el servidor **FastMCP** y registramos las **5 herramientas**. FastMCP genera automáticamente el esquema de cada una a partir de sus *type hints* y *docstrings*.


In [46]:
from fastmcp import FastMCP

mcp = FastMCP("Servidor de Monedas y Clima")

# Registramos las 5 herramientas (funciones definidas en las celdas anteriores)
for _fn in (convert_currency, get_exchange_rates,
            geocode_city, get_current_weather, get_weather_forecast):
    mcp.tool(_fn)

print("✅ Servidor MCP creado con las herramientas:")
for _n in ("convert_currency", "get_exchange_rates", "geocode_city",
           "get_current_weather", "get_weather_forecast"):
    print("   •", _n)


✅ Servidor MCP creado con las herramientas:
   • convert_currency
   • get_exchange_rates
   • geocode_city
   • get_current_weather
   • get_weather_forecast


## 8. Arrancar el servidor en localhost (`main_server.py`)

Levantamos el servidor MCP en `localhost:8000` en un **hilo en segundo plano** (con HTTP *streamable*), para que el cliente pueda conectarse como `http://127.0.0.1:8000/mcp`. En un proyecto real, `main_server.py` haría simplemente `mcp.run(transport="http", host="127.0.0.1", port=8000)`.


In [47]:
import asyncio
import threading
import time
import socket

if "_hilo_servidor" not in globals():
    _hilo_servidor = None

_error_servidor = None


def _arrancar_servidor():
    global _error_servidor
    try:
        # asyncio.run() crea un event loop propio en el hilo,
        # lo que permite a anyio/sniffio detectar el backend asyncio correctamente.
        asyncio.run(mcp.run_http_async(
            host=MCP_HOST, port=MCP_PORT, log_level="warning", show_banner=False
        ))
    except Exception as e:
        _error_servidor = e


def _servidor_listo(host, port, intentos=10, espera=0.5):
    """Comprueba que el servidor acepta conexiones TCP."""
    for _ in range(intentos):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(0.5)
            if s.connect_ex((host, port)) == 0:
                return True
        time.sleep(espera)
    return False


if _hilo_servidor is None or not _hilo_servidor.is_alive():
    _error_servidor = None
    _hilo_servidor = threading.Thread(target=_arrancar_servidor, daemon=True)
    _hilo_servidor.start()
    if _servidor_listo(MCP_HOST, MCP_PORT):
        print(f"✅ Servidor MCP escuchando en {MCP_URL}")
    else:
        msg = str(_error_servidor) if _error_servidor else "timeout: el servidor no respondió"
        print(f"❌ Servidor MCP NO arrancado: {msg}")
else:
    print(f"ℹ️  El servidor MCP ya estaba en marcha en {MCP_URL}")


ℹ️  El servidor MCP ya estaba en marcha en http://127.0.0.1:8000/mcp


## 9. Cliente OpenAI con MCP (`client/openai_client.py`)

El cliente se conecta al **servidor MCP local**, obtiene sus herramientas y las expone a OpenAI como *function tools*. Implementa el bucle de *tool calling* (que permite **llamadas secuenciales**: primero `geocode_city`, luego `get_current_weather`), con **reintentos** ante fallos de conexión y manejo de errores de las herramientas.

> Nota: la vía "MCP hospedado" de la *Responses API* (`type: "mcp"`) requiere que el servidor sea accesible públicamente; como aquí es **local**, se puentean las herramientas MCP al *function calling* de **Chat Completions** (permitido por el enunciado), conectándose igualmente al servidor MCP local.


In [48]:
import json
import time
import logging
from typing import Any
from fastmcp import Client
from openai import OpenAI, RateLimitError, APIConnectionError, APITimeoutError

cliente_logger = logging.getLogger("openai_client")

# Modelos permitidos por el enunciado (se admite cualquier variante de estas familias)
_FAMILIAS_MODELO_PERMITIDAS = ("gpt-4o", "gpt-4.1", "gpt-5")

SYSTEM_PROMPT = (
    "Eres un asistente que ayuda con conversión de divisas y con el clima por ciudad. "
    "Dispones de herramientas (MCP) para ello. Para el tiempo de una ciudad, primero usa "
    "geocode_city para obtener coordenadas y luego get_current_weather o get_weather_forecast. "
    "Responde SIEMPRE en español, de forma clara y con los datos reales que devuelvan las "
    "herramientas. Si una herramienta falla, explica el problema con naturalidad."
)


class MCPClienteOpenAI:
    """Cliente OpenAI que usa las herramientas del servidor MCP local."""

    def __init__(self, mcp_target=None, base_url=None, api_key=None, modelo=None,
                 system_prompt=SYSTEM_PROMPT, max_reintentos=None, max_pasos=None,
                 temperature=None, top_p=None, max_tokens=None):
        # mcp_target: URL del servidor MCP (str) o el objeto `mcp` (en memoria)
        self.mcp_target = mcp_target if mcp_target is not None else MCP_URL
        self.oai = OpenAI(base_url=base_url or OPENAI_BASE_URL, api_key=api_key or OPENAI_API_KEY)
        self.modelo = modelo or OPENAI_MODEL
        # Validación: el enunciado exige gpt-4o / gpt-4.1 / gpt-5 (o variantes de esas familias)
        if not any(self.modelo.startswith(f) for f in _FAMILIAS_MODELO_PERMITIDAS):
            raise ValueError(
                f"Modelo '{self.modelo}' no permitido. Usa una de las familias "
                f"{_FAMILIAS_MODELO_PERMITIDAS} (p. ej. gpt-5). Ajusta OPENAI_MODEL en el .env."
            )
        self.system_prompt = system_prompt
        # Si no se pasan explícitamente, se toman de la configuración por entorno (celda 4)
        self.max_reintentos = max_reintentos if max_reintentos is not None else MAX_REINTENTOS_OPENAI
        self.max_pasos = max_pasos if max_pasos is not None else MAX_PASOS_HERRAMIENTAS
        # Parámetros de generación opcionales (solo se envían si se definen).
        # Nota: algunos modelos gpt-5 de razonamiento no aceptan 'temperature'; por eso van a None.
        self.temperature = temperature
        self.top_p = top_p
        self.max_tokens = max_tokens
        self.historial: list[dict[str, Any]] = []

    async def _tools_para_openai(self, client) -> list[dict[str, Any]]:
        tools = await client.list_tools()
        return [{
            "type": "function",
            "function": {
                "name": t.name,
                "description": t.description or "",
                "parameters": t.inputSchema,
            },
        } for t in tools]

    @staticmethod
    def _texto_resultado(result) -> str:
        """Extrae texto/JSON del resultado de una herramienta MCP (varias versiones)."""
        data = getattr(result, "data", None)
        if data is not None:
            return json.dumps(data, ensure_ascii=False, default=str)
        structured = getattr(result, "structured_content", None)
        if structured is not None:
            return json.dumps(structured, ensure_ascii=False, default=str)
        contenido = getattr(result, "content", None) or []
        textos = [getattr(b, "text", "") for b in contenido if getattr(b, "text", "")]
        return "\n".join(textos) if textos else str(result)

    def _chat(self, mensajes, herramientas):
        """Llamada a Chat Completions con reintentos ante fallos de conexión."""
        kwargs: dict[str, Any] = dict(
            model=self.modelo, messages=mensajes,
            tools=herramientas, tool_choice="auto",
        )
        if self.temperature is not None:
            kwargs["temperature"] = self.temperature
        if self.top_p is not None:
            kwargs["top_p"] = self.top_p
        if self.max_tokens is not None:
            kwargs["max_tokens"] = self.max_tokens

        # Errores de red/límite: merece la pena reintentar. Otros (p. ej. 400): no.
        errores_reintentables = (RateLimitError, APIConnectionError, APITimeoutError)
        ultimo_error = None
        for intento in range(self.max_reintentos + 1):
            try:
                return self.oai.chat.completions.create(**kwargs)
            except errores_reintentables as e:
                ultimo_error = e
                # RateLimitError: esperamos algo más antes de reintentar
                espera = 3.0 * (intento + 1) if isinstance(e, RateLimitError) else 1.5 * (intento + 1)
                cliente_logger.warning("Error de red/límite con OpenAI (intento %d/%d): %s: %s",
                                       intento + 1, self.max_reintentos + 1, type(e).__name__, e)
                if intento < self.max_reintentos:
                    time.sleep(espera)
            except Exception as e:
                # Error no recuperable (autenticación, petición inválida, etc.): no reintentar
                cliente_logger.error("Error no recuperable con OpenAI: %s: %s", type(e).__name__, e)
                raise
        raise ultimo_error

    async def ask(self, pregunta: str) -> str:
        """Procesa una consulta en lenguaje natural usando las herramientas MCP."""
        async with Client(self.mcp_target) as client:
            herramientas = await self._tools_para_openai(client)
            mensajes = ([{"role": "system", "content": self.system_prompt}]
                        + self.historial
                        + [{"role": "user", "content": pregunta}])

            for _ in range(self.max_pasos):
                respuesta = self._chat(mensajes, herramientas)
                msg = respuesta.choices[0].message

                asistente = {"role": "assistant", "content": msg.content or ""}
                if msg.tool_calls:
                    asistente["tool_calls"] = [{
                        "id": tc.id, "type": "function",
                        "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                    } for tc in msg.tool_calls]
                mensajes.append(asistente)

                if not msg.tool_calls:
                    self.historial += [{"role": "user", "content": pregunta},
                                       {"role": "assistant", "content": msg.content or ""}]
                    return msg.content or ""

                # Ejecutar cada herramienta solicitada vía MCP
                for tc in msg.tool_calls:
                    try:
                        args = json.loads(tc.function.arguments or "{}")
                        cliente_logger.info("Ejecutando herramienta %s con %s", tc.function.name, args)
                        resultado = await client.call_tool(tc.function.name, args)
                        contenido = self._texto_resultado(resultado)
                    except Exception as e:
                        cliente_logger.warning("Error en herramienta %s: %s", tc.function.name, e)
                        contenido = json.dumps({"error": f"{type(e).__name__}: {e}"}, ensure_ascii=False)
                    mensajes.append({"role": "tool", "tool_call_id": tc.id, "content": contenido})

            return "(Se alcanzó el número máximo de pasos de herramientas sin respuesta final.)"

    def reset(self):
        self.historial.clear()


## 10. Interfaz CLI (`client/cli_interface.py` / `main_client.py`)

CLI interactiva en lenguaje natural con comandos especiales: **`/salir`**, **`/ayuda`** y **`/monedas`**. Maneja errores de entrada y ciudades no encontradas (los gestiona el cliente).


In [49]:
def _ayuda():
    print("\nℹ️  Puedo ayudarte con DIVISAS y CLIMA. Ejemplos:")
    print("   • Convierte 100 USD a EUR")
    print("   • ¿Cuál es el clima actual en Madrid?")
    print("   • Dame el pronóstico del tiempo para Nueva York")
    print("   • ¿Qué coordenadas tiene Tokio?")
    print("Comandos:  /ayuda  ·  /monedas  ·  /salir\n")


def _monedas():
    print("\n💱 Códigos de moneda habituales (ISO 4217):")
    print("   " + ", ".join(MONEDAS_COMUNES))
    print("   (ExchangeRate-API admite muchas más; usa el código de 3 letras.)\n")


async def run_cli(cliente: "MCPClienteOpenAI | None" = None):
    """Bucle interactivo de la CLI del cliente OpenAI + MCP."""
    print("=" * 62)
    print("🛠️  Asistente de Divisas y Clima (OpenAI + MCP)")
    print("=" * 62)

    if not OPENAI_API_KEY:
        print("❌ Falta AZURE_OPENAI_API_KEY en el .env.")
        return

    # Diagnóstico: modelo y endpoint en uso (útil con varias configuraciones)
    print(f"🧠 Modelo OpenAI: {OPENAI_MODEL}")
    print(f"🔗 Endpoint:      {OPENAI_BASE_URL}")
    print(f"🔌 Servidor MCP:  {MCP_URL}")

    # Chequeo de salud de ExchangeRate-API (llamada de prueba ligera)
    operativa, detalle = salud_exchange()
    if operativa:
        print("💱 Divisas: ✅ ExchangeRate-API operativa")
    else:
        print(f"💱 Divisas: ❌ no disponibles ({detalle}). El clima sí funciona.")
    print("🌦️  Clima: ✅ disponible (Open-Meteo, sin clave)")

    cliente = cliente or MCPClienteOpenAI()
    _ayuda()

    while True:
        try:
            consulta = input("🧑 Tú > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 ¡Hasta luego!")
            break

        if not consulta:
            continue
        comando = consulta.lower()
        if comando in ("/salir", "/exit", "/quit", "salir"):
            print("👋 ¡Hasta luego!")
            break
        if comando in ("/ayuda", "/help"):
            _ayuda()
            continue
        if comando in ("/monedas", "/currencies"):
            _monedas()
            continue

        try:
            respuesta = await cliente.ask(consulta)
            print(f"\n🤖 {respuesta}\n")
        except Exception as e:
            print(f"⚠️  No se pudo procesar la consulta ({type(e).__name__}: {e}). Inténtalo de nuevo.\n")


### ▶️ Opción A — Consultas directas (sin `input()`, recomendado en notebooks)

Crea el cliente y hazle preguntas con `await cliente.ask(...)`. Demuestra el flujo completo: conversión de divisas y geocodificación → clima.

> Requiere haber arrancado el servidor (sección 8). Para divisas necesitas `API_KEY_EXCHANGE`; el clima funciona sin clave.


In [50]:
cliente = MCPClienteOpenAI()

for consulta in [
    "¿Qué coordenadas tiene Tokio?",
    "¿Cuál es el clima actual en Madrid?",
    "Convierte 100 USD a EUR",
    "Dame las tasas de cambio del euro frente a otras monedas",
]:
    print(f"🧑 {consulta}")
    print(f"🤖 {await cliente.ask(consulta)}\n")


🧑 ¿Qué coordenadas tiene Tokio?


2026-07-01 12:11:09,055 [INFO] mcp.server.streamable_http_manager: Created new transport with session ID: fe2b6b29c10b4d639790ea1880ced608
2026-07-01 12:11:09,058 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:11:09,058 [INFO] mcp.client.streamable_http: Received session ID: fe2b6b29c10b4d639790ea1880ced608
2026-07-01 12:11:09,060 [INFO] mcp.client.streamable_http: Negotiated protocol version: 2025-11-25
2026-07-01 12:11:09,073 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 202 Accepted"
2026-07-01 12:11:09,074 [INFO] httpx: HTTP Request: GET http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:11:09,078 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:11:09,079 [INFO] mcp.server.lowlevel.server: Processing request of type ListToolsRequest
2026-07-01 12:11:14,969 [INFO] httpx: HTTP Request: POST https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/chat/co

🤖 Las coordenadas de Tokio son:
- Latitud: 35.6895
- Longitud: 139.69171
País: Japón
Huso horario: Asia/Tokyo

🧑 ¿Cuál es el clima actual en Madrid?


2026-07-01 12:11:17,161 [INFO] mcp.server.streamable_http_manager: Created new transport with session ID: a2eccc543aed4488bc759e6cc35e8e7a
2026-07-01 12:11:17,163 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:11:17,164 [INFO] mcp.client.streamable_http: Received session ID: a2eccc543aed4488bc759e6cc35e8e7a
2026-07-01 12:11:17,164 [INFO] mcp.client.streamable_http: Negotiated protocol version: 2025-11-25
2026-07-01 12:11:17,168 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 202 Accepted"
2026-07-01 12:11:17,169 [INFO] httpx: HTTP Request: GET http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:11:17,174 [INFO] mcp.server.lowlevel.server: Processing request of type ListToolsRequest
2026-07-01 12:11:17,175 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:11:20,805 [INFO] httpx: HTTP Request: POST https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/chat/co

🤖 Clima actual en Madrid (Europe/Madrid)
- Temperatura: 29.6 °C
- Humedad: 35 %
- Viento: 10.2 km/h
- Cielo: Mayormente despejado

🧑 Convierte 100 USD a EUR


2026-07-01 12:11:25,261 [INFO] mcp.client.streamable_http: Received session ID: 041c47d132f448b888f9a8795cc6b513
2026-07-01 12:11:25,262 [INFO] mcp.client.streamable_http: Negotiated protocol version: 2025-11-25
2026-07-01 12:11:25,273 [INFO] httpx: HTTP Request: GET http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:11:25,274 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 202 Accepted"
2026-07-01 12:11:25,280 [INFO] mcp.server.lowlevel.server: Processing request of type ListToolsRequest
2026-07-01 12:11:25,281 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:11:29,013 [INFO] httpx: HTTP Request: POST https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-01 12:11:29,015 [INFO] openai_client: Ejecutando herramienta convert_currency con {'amount': 100, 'from_currency': 'USD', 'to_currency': 'EUR'}
2026-07-01 12:11:29,031 [INFO] httpx: HTTP Request: POST htt

🤖 100 USD equivalen a 87.66 EUR (tasa: 1 USD = 0.8766 EUR).

🧑 Dame las tasas de cambio del euro frente a otras monedas


2026-07-01 12:11:31,156 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:11:31,157 [INFO] mcp.client.streamable_http: Received session ID: 5f46c4a84ac04b2d93732c4cff9cd9e6
2026-07-01 12:11:31,158 [INFO] mcp.client.streamable_http: Negotiated protocol version: 2025-11-25
2026-07-01 12:11:31,165 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 202 Accepted"
2026-07-01 12:11:31,166 [INFO] httpx: HTTP Request: GET http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:11:31,177 [INFO] mcp.server.lowlevel.server: Processing request of type ListToolsRequest
2026-07-01 12:11:31,178 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:11:34,019 [INFO] httpx: HTTP Request: POST https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-01 12:11:34,021 [INFO] openai_client: Ejecutando herramienta get_exchange_rates con {'base_currency': '

🤖 Aquí tienes algunas tasas actuales con base 1 EUR (según la herramienta):

- USD: 1.1411
- GBP: 0.8614
- JPY: 185.464
- CHF: 0.9230
- CNY: 7.7547
- HKD: 8.9513
- CAD: 1.6215
- AUD: 1.6529
- NZD: 2.0124
- SGD: 1.4770
- SEK: 11.0838
- NOK: 11.3134
- DKK: 7.4635
- PLN: 4.2974
- CZK: 24.2547
- HUF: 355.7777
- RON: 5.2440
- TRY: 53.2568
- RUB: 89.4053
- MXN: 19.9469
- BRL: 5.9010
- ARS: 1691.8237
- CLP: 1051.5234
- COP: 3936.5451
- PEN: 3.8929
- ZAR: 18.6986
- AED: 4.1896
- SAR: 4.2780
- KRW: 1767.5178
- INR: 107.9661

Si necesitas otra moneda específica o el listado completo, dímelo y te lo facilito. También puedo convertir una cantidad concreta.



### ▶️ Opción B — CLI interactiva con `run_cli()`


In [51]:
await run_cli()


2026-07-01 12:11:53,832 [INFO] api_clients: GET https://v6.exchangerate-api.com/v6/***/latest/USD params={}


🛠️  Asistente de Divisas y Clima (OpenAI + MCP)
🧠 Modelo OpenAI: gpt-5
🔗 Endpoint:      https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/
🔌 Servidor MCP:  http://127.0.0.1:8000/mcp
💱 Divisas: ✅ ExchangeRate-API operativa
🌦️  Clima: ✅ disponible (Open-Meteo, sin clave)

ℹ️  Puedo ayudarte con DIVISAS y CLIMA. Ejemplos:
   • Convierte 100 USD a EUR
   • ¿Cuál es el clima actual en Madrid?
   • Dame el pronóstico del tiempo para Nueva York
   • ¿Qué coordenadas tiene Tokio?
Comandos:  /ayuda  ·  /monedas  ·  /salir



2026-07-01 12:12:34,911 [INFO] mcp.server.streamable_http_manager: Created new transport with session ID: 6ba505e30cc543b89e39d4eefa16e1d4
2026-07-01 12:12:34,913 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:12:34,914 [INFO] mcp.client.streamable_http: Received session ID: 6ba505e30cc543b89e39d4eefa16e1d4
2026-07-01 12:12:34,915 [INFO] mcp.client.streamable_http: Negotiated protocol version: 2025-11-25
2026-07-01 12:12:34,922 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 202 Accepted"
2026-07-01 12:12:34,923 [INFO] httpx: HTTP Request: GET http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:12:34,929 [INFO] mcp.server.lowlevel.server: Processing request of type ListToolsRequest
2026-07-01 12:12:34,930 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:12:38,501 [INFO] httpx: HTTP Request: POST https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/chat/co


🤖 100 USD equivalen a 87.66 EUR con una tasa de 1 USD = 0.8766 EUR.



2026-07-01 12:12:49,332 [INFO] mcp.server.streamable_http_manager: Created new transport with session ID: 965611c3f57c4f6eb7e2233bb229686a
2026-07-01 12:12:49,334 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:12:49,336 [INFO] mcp.client.streamable_http: Received session ID: 965611c3f57c4f6eb7e2233bb229686a
2026-07-01 12:12:49,338 [INFO] mcp.client.streamable_http: Negotiated protocol version: 2025-11-25
2026-07-01 12:12:49,343 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 202 Accepted"
2026-07-01 12:12:49,344 [INFO] httpx: HTTP Request: GET http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:12:49,346 [INFO] mcp.server.lowlevel.server: Processing request of type ListToolsRequest
2026-07-01 12:12:49,347 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:12:53,543 [INFO] httpx: HTTP Request: POST https://marcvancutseme7172-2656-resource.services.ai.azure.com/openai/v1/chat/co


🤖 Tokio, Japón
- Coordenadas: 35.6895, 139.69171
- Zona horaria: Asia/Tokyo


💱 Códigos de moneda habituales (ISO 4217):
   USD, EUR, GBP, JPY, CHF, CAD, AUD, CNY, MXN, BRL, ARS, INR, RUB, SEK, NOK, PLN
   (ExchangeRate-API admite muchas más; usa el código de 3 letras.)


ℹ️  Puedo ayudarte con DIVISAS y CLIMA. Ejemplos:
   • Convierte 100 USD a EUR
   • ¿Cuál es el clima actual en Madrid?
   • Dame el pronóstico del tiempo para Nueva York
   • ¿Qué coordenadas tiene Tokio?
Comandos:  /ayuda  ·  /monedas  ·  /salir

👋 ¡Hasta luego!


## 11. (Opcional) Probar solo el servidor MCP (sin OpenAI)

Comprueba que el servidor expone las 5 herramientas y que el flujo geocodificación → clima funciona, llamando a las herramientas directamente con el cliente MCP (sin gastar tokens de OpenAI).


In [52]:
from fastmcp import Client

async with Client(MCP_URL) as client:
    tools = await client.list_tools()
    print("Herramientas disponibles:", [t.name for t in tools])

    # Flujo: geocodificación -> clima
    geo = await client.call_tool("geocode_city", {"city": "Madrid"})
    print("\ngeocode_city('Madrid') ->", MCPClienteOpenAI._texto_resultado(geo))

    g = getattr(geo, "data", None) or {}
    if g.get("latitude") is not None:
        clima = await client.call_tool("get_current_weather",
                                       {"latitude": g["latitude"], "longitude": g["longitude"]})
        print("get_current_weather(...) ->", MCPClienteOpenAI._texto_resultado(clima))


2026-07-01 12:13:07,606 [INFO] mcp.server.streamable_http_manager: Created new transport with session ID: 74cbce202aec4c72a10b3b568efcd60c
2026-07-01 12:13:07,608 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:13:07,609 [INFO] mcp.client.streamable_http: Received session ID: 74cbce202aec4c72a10b3b568efcd60c
2026-07-01 12:13:07,609 [INFO] mcp.client.streamable_http: Negotiated protocol version: 2025-11-25
2026-07-01 12:13:07,614 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 202 Accepted"
2026-07-01 12:13:07,614 [INFO] httpx: HTTP Request: GET http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:13:07,618 [INFO] mcp.server.lowlevel.server: Processing request of type ListToolsRequest
2026-07-01 12:13:07,619 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:13:07,630 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:13:07,630 [INFO] mcp.s

Herramientas disponibles: ['convert_currency', 'get_exchange_rates', 'geocode_city', 'get_current_weather', 'get_weather_forecast']


2026-07-01 12:13:08,015 [INFO] mcp.server.lowlevel.server: Processing request of type CallToolRequest
2026-07-01 12:13:08,015 [INFO] httpx: HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:13:08,017 [INFO] api_clients: get_current_weather: 40.4165, -3.7026
2026-07-01 12:13:08,018 [INFO] api_clients: GET https://api.open-meteo.com/v1/forecast params={'latitude': 40.4165, 'longitude': -3.7026, 'current': 'temperature_2m,relative_humidity_2m,weather_code,wind_speed_10m', 'timezone': 'auto'}



geocode_city('Madrid') -> {"city": "Madrid", "country": "España", "latitude": 40.4165, "longitude": -3.70256, "timezone": "Europe/Madrid"}


2026-07-01 12:13:08,413 [INFO] mcp.server.streamable_http: Terminating session: 74cbce202aec4c72a10b3b568efcd60c
2026-07-01 12:13:08,414 [INFO] httpx: HTTP Request: DELETE http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"
2026-07-01 12:13:08,415 [INFO] mcp.client.streamable_http: GET stream disconnected, reconnecting in 1000ms...


get_current_weather(...) -> {"temperature_c": 29.6, "humidity_pct": 35, "wind_kmh": 10.2, "weather_code": 1, "description": "Mayormente despejado"}


## 12. Pruebas automatizadas (`pytest`)

Pruebas de regresión que verifican los flujos clave: geocodificación, geocodificación → clima, normalización del pronóstico, conversión de divisas (se omite si no hay `API_KEY_EXCHANGE`) y validación de códigos de moneda. Usan las funciones de herramienta directamente (son síncronas) con `ipytest`.

In [53]:
# Pruebas automatizadas con ipytest/pytest.
# Verifican flujos clave llamando a las funciones de herramienta (síncronas):
#   - geocodificación,  geocodificación → clima,  conversión de divisas,  validación.
try:
    import ipytest
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipytest", "pytest"], check=True)
    import ipytest

import pytest
ipytest.autoconfig()


def test_geocode_devuelve_coordenadas():
    r = geocode_city("Madrid")
    assert -90 <= r["latitude"] <= 90
    assert -180 <= r["longitude"] <= 180
    assert r["country"]


def test_flujo_geocodificacion_a_clima():
    g = geocode_city("Madrid")
    w = get_current_weather(g["latitude"], g["longitude"])
    assert w["temperature_c"] is not None
    assert isinstance(w["description"], str)


def test_pronostico_normaliza_dias():
    g = geocode_city("Tokio")
    f = get_weather_forecast(g["latitude"], g["longitude"], days=99)  # se ajusta a 16
    assert 1 <= len(f["days"]) <= 16


def test_conversion_divisas():
    if not API_KEY_EXCHANGE:
        pytest.skip("Sin API_KEY_EXCHANGE: se omite la prueba de divisas.")
    r = convert_currency(100, "USD", "EUR")
    assert r["converted_amount"] > 0
    assert r["from_currency"] == "USD" and r["to_currency"] == "EUR"


def test_validacion_moneda_invalida():
    # No requiere red: valida el código antes de llamar a la API
    with pytest.raises(ValueError):
        convert_currency(100, "US", "EUR")


ipytest.run("-q")


2026-07-01 12:13:15,414 [INFO] api_clients: geocode_city: Madrid
2026-07-01 12:13:15,416 [INFO] api_clients: GET https://geocoding-api.open-meteo.com/v1/search params={'name': 'Madrid', 'count': 1, 'language': 'es', 'format': 'json'}


.

2026-07-01 12:13:15,927 [INFO] api_clients: geocode_city: Madrid
2026-07-01 12:13:15,928 [INFO] api_clients: GET https://geocoding-api.open-meteo.com/v1/search params={'name': 'Madrid', 'count': 1, 'language': 'es', 'format': 'json'}
2026-07-01 12:13:16,357 [INFO] api_clients: get_current_weather: 40.4165, -3.7026
2026-07-01 12:13:16,357 [INFO] api_clients: GET https://api.open-meteo.com/v1/forecast params={'latitude': 40.4165, 'longitude': -3.7026, 'current': 'temperature_2m,relative_humidity_2m,weather_code,wind_speed_10m', 'timezone': 'auto'}


.

2026-07-01 12:13:16,796 [INFO] api_clients: geocode_city: Tokio
2026-07-01 12:13:16,797 [INFO] api_clients: GET https://geocoding-api.open-meteo.com/v1/search params={'name': 'Tokio', 'count': 1, 'language': 'es', 'format': 'json'}
2026-07-01 12:13:17,227 [INFO] api_clients: get_weather_forecast: 35.6895, 139.6917 (16 días)
2026-07-01 12:13:17,228 [INFO] api_clients: GET https://api.open-meteo.com/v1/forecast params={'latitude': 35.6895, 'longitude': 139.6917, 'daily': 'temperature_2m_max,temperature_2m_min,weather_code', 'forecast_days': 16, 'timezone': 'auto'}


.

2026-07-01 12:13:17,650 [INFO] api_clients: convert_currency: 100.0 USD -> EUR
2026-07-01 12:13:17,650 [INFO] api_clients: GET https://v6.exchangerate-api.com/v6/***/pair/USD/EUR/100.0 params={}


..                                                                                        [100%]
======================================== warnings summary =========================================
..\..\..\..\..\AppData\Roaming\Python\Python313\site-packages\_pytest\config\__init__.py:1345
  C:\Users\macdu\AppData\Roaming\Python\Python313\site-packages\_pytest\config\__init__.py:1345: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html


<ExitCode.OK: 0>